# Tutorial 01: Function Fitting and Its Limits

Before diving into PINNs, let's build intuition by fitting a simple function with a plain neural network. Our target is the sine function:

$$y = \sin(x), \quad x \in [-2\pi,\; 2\pi]$$

We'll use a multi-layer perceptron (MLP) to learn this mapping from data alone — no physics, just input-output pairs. Along the way, we'll introduce the basic Talos workflow:

1. **Prepare data** — wrap numpy arrays into a `ta.Dataset` and split into train/test sets
2. **Build model** — create an MLP from the Talos model zoo
3. **Train & evaluate** — use `TorchTrainer` to optimize, then visualize results

At the end, we'll see what happens when the network is trained on only *half* the domain — a failure that motivates the PINN approach in the next tutorial.

### Step 0: Setups
---

In [ ]:
# To allow automatically uses the newest version of your code.  
%load_ext autoreload
%autoreload 2

In [ ]:
# Add necessary paths to the system path
from utils import add_necessary_paths
add_necessary_paths()

# Import libraries
from utils import check_torch

import utils.u01 as u
import talos as ta
import numpy as np

# Do sanity checks before running the code
check_torch()

# Fix random seed for reproducibility
# NOTE: Run cells IN ORDER to ensure deterministic behavior.
ta.set_seed()

### Step 1: Prepare data
---

Talos manages data through `ta.Dataset` — a lightweight wrapper around numpy arrays that provides splitting, shuffling, sampling, and reporting. Here we:

- Generate 200 evenly spaced points over $[-2\pi, 2\pi]$
- Compute $y = \sin(x)$ as labels
- Split 50/50 into train and test sets with **shuffling** (so both sets cover the full domain)

In [ ]:
# configure data set  
t_min, t_max = -2 * np.pi, 2 * np.pi
n_points = 200
shuffle = 1
train_test_ratio = (1, 1)

# Generate data
X = np.linspace(t_min, t_max, n_points).reshape(-1, 1)  # Reshape to a column vector
Y = np.sin(X).reshape(-1, 1)  # Reshape to a column vector
dataset = ta.Dataset(X, Y, name='sin(x)')
dataset.report()

# Split data
train_set, test_set = dataset.split(*train_test_ratio, shuffle=shuffle)
train_set.report()
test_set.report()

# Visualize data
u.plot_data(train_set, test_set, title=f'Dataset Visualization (shuffle = {shuffle})')

### Step 2: Build model
---

Talos provides a model zoo with pre-built architectures. `ta.model.torch_zoo.MLP` creates a configurable multi-layer perceptron. Our network:

- **1** input feature ($x$) → **5** hidden layers `[50, 50, 20, 50, 50]` → **1** output ($\hat{y}$)
- **tanh** activation — a smooth, bounded function well-suited for fitting periodic targets

In [ ]:
# Model configuration  
hidden_features = [50, 50, 20, 50, 50]

# Instantiate a model
model = ta.model.torch_zoo.MLP(1, hidden_features, 1, activation='tanh')
print(model)

### Step 3: Train and evaluate
---

Training in Talos follows a simple pattern:

1. Create a `TorchTrainer` with a model and loss function
2. Configure training options (early stopping, validation, etc.)
3. Call `trainer.train(dataset, max_iterations)`

The trainer handles the full loop: batching, forward/backward passes, validation, and early stopping. After training, we use `model.predict()` to generate predictions and compare against ground truth.

In [ ]:
# Configure training parameters  
trainer = u.get_trainer(model, loss_fn='mse', early_stop=True, patience=2,
                        val_ratio=0.1, validate_every=10000, print_every=5000)

# Train the model  
trainer.train(train_set, 50000)

In [ ]:
# Evaluate the model on the test set
Y_pred = model.predict(test_set)

# Visualize the model prediction against the ground truth
u.plot_data(train_set, test_set, Y_pred, eval=True)

### Step 4: The generalization problem
---

The model above works well because shuffling mixes points from the entire domain into both train and test sets. But what if the model only sees the **left half** $[-2\pi, 0]$ during training?

With `shuffle=False` and a 1:1 split, the first 100 points (left half) become the training set and the last 100 (right half) become the test set. The network must **extrapolate** — predict in a region it has never seen. Neural networks are notoriously bad at this.

In [ ]:
# Re-split dataset without shuffling and visualize
train_set_ns, test_set_ns = dataset.split(*train_test_ratio, shuffle=False)

u.plot_data(train_set_ns, test_set_ns, title=f'Dataset Visualization (shuffle = {shuffle})')

In [ ]:
# Initialize another model and train it
model_alt = ta.model.torch_zoo.MLP(1, hidden_features, 1, activation='tanh')

trainer_alt = u.get_trainer(model_alt, loss_fn='mse', early_stop=True, patience=2,
                             val_ratio=0.1, validate_every=10000, print_every=5000)

trainer_alt.train(train_set_ns, 50000)

In [ ]:
# Evaluate and visualize
Y_pred_alt = model_alt.predict(test_set_ns)
u.plot_data(train_set_ns, test_set_ns, Y_pred_alt, eval=True,
            title='Alt Model Prediction vs Ground Truth (Test Set (ns))')

### Takeaway

The MLP fits $\sin(x)$ perfectly when train and test data are drawn from the same domain (shuffled split). But when asked to extrapolate beyond its training region, it fails — the network has learned a curve that happens to match the data, not the underlying *law* that generates it.

In the **next tutorial**, we'll fix this by telling the network about the governing ODE $y' = \cos(x)$. This is the idea behind **Physics-Informed Neural Networks (PINNs)**.